In [0]:
import os
from soda.scan import Scan

def run_soda_silver_layer(spark, df_sessions, df_laps, yaml_filepath):
    """
    Executes a single SodaCL scan for the entire Silver layer.
    """
    print(f"\n{'='*50}")
    print(f"🚀 RUNNING SODA CHECKS FOR: SILVER LAYER")
    print(f"{'='*50}")

    # 1. Register BOTH DataFrames as temp views using safe names
    df_sessions.createOrReplaceTempView("silver_cleaned_sessions")
    df_laps.createOrReplaceTempView("silver_enriched_laps")

    # 2. Read the YAML file
    # Pro-tip: Using os.getcwd() is safer for Databricks Repos/Workspaces
    repo_root = "/Workspace/Users/jafar.gahramanov@gmail.com/F1-Pulse" 
    full_yaml_path = f"{repo_root}/{yaml_filepath}"

    with open(full_yaml_path, "r") as file:
        yaml_content = file.read()

    # 3. Patch BOTH table names in the YAML to match the Temp Views
    yaml_content = yaml_content.replace(
        "checks for silver.cleaned_sessions", 
        "checks for silver_cleaned_sessions"
    )
    yaml_content = yaml_content.replace(
        "checks for silver.enriched_laps", 
        "checks for silver_enriched_laps"
    )

    # 4. Configure and run Soda
    scan = Scan()
    scan.set_data_source_name("spark_df")
    scan.add_spark_session(spark)
    scan.add_sodacl_yaml_str(yaml_content)

    exit_code = scan.execute()

    # 5. Output results
    print(scan.get_logs_text())

    if exit_code != 0:
        print(f"❌ SODA DQ CHECKS FAILED FOR SILVER LAYER")
    else:
        print(f"✅ SODA DQ CHECKS PASSED FOR SILVER LAYER")

    return exit_code

# =========================================
# Run the check
# =========================================
run_soda_silver_layer(
    spark=spark,
    df_sessions=df_silver_sessions,
    df_laps=df_silver_laps,
    yaml_filepath="soda/checks_silver.yml"
)

In [0]:
def run_soda_gold_layer(spark, df_driver_metrics, df_constructors, df_progression, yaml_filepath):
    """
    Executes a single SodaCL scan for the entire Gold layer.
    """
    print(f"\n{'='*50}")
    print(f"🚀 RUNNING SODA CHECKS FOR: GOLD LAYER")
    print(f"{'='*50}")

    # 1. Register ALL THREE DataFrames as temp views
    df_driver_metrics.createOrReplaceTempView("gold_driver_performance_metrics")
    df_constructors.createOrReplaceTempView("gold_constructor_standings")
    df_progression.createOrReplaceTempView("gold_lap_progression")

    # 2. Read the YAML file
    repo_root = "/Workspace/Users/jafar.gahramanov@gmail.com/F1-Pulse" 
    full_yaml_path = f"{repo_root}/{yaml_filepath}"

    with open(full_yaml_path, "r") as file:
        yaml_content = file.read()

    # 3. Patch all three table names in the YAML to match the Temp Views
    # (Note: If you change the YAML file to use underscores natively, you can delete these three lines)
    yaml_content = yaml_content.replace("checks for gold.driver_performance_metrics", "checks for gold_driver_performance_metrics")
    yaml_content = yaml_content.replace("checks for gold.constructor_standings", "checks for gold_constructor_standings")
    yaml_content = yaml_content.replace("checks for gold.lap_progression", "checks for gold_lap_progression")

    # 4. Configure and run Soda
    scan = Scan()
    scan.set_data_source_name("spark_df")
    scan.add_spark_session(spark)
    scan.add_sodacl_yaml_str(yaml_content)

    exit_code = scan.execute()

    # 5. Output results
    print(scan.get_logs_text())

    if exit_code != 0:
        print(f"❌ SODA DQ CHECKS FAILED FOR GOLD LAYER")
    else:
        print(f"✅ SODA DQ CHECKS PASSED FOR GOLD LAYER")

    return exit_code

In [0]:
# =========================================
# ORCHESTRATION: Run Silver, then Gold
# =========================================

# 1. Run Silver
silver_status = run_soda_silver_layer(
    spark=spark,
    df_sessions=df_silver_sessions,
    df_laps=df_silver_laps,
    yaml_filepath="soda/checks_silver.yml"
)

# 2. Check if Silver passed before running Gold
if silver_status == 0:
    print("✅ Silver checks passed. Proceeding to Gold...")
    
    gold_status = run_soda_gold_layer(
        spark=spark,
        df_driver_metrics=df_gold_leaderboard,
        df_constructors=df_gold_constructors,
        df_progression=df_gold_progression,
        yaml_filepath="soda/checks_gold.yml"
    )
    
    if gold_status != 0:
        # Fails the Databricks notebook task if Gold fails
        raise Exception("Pipeline halted: Gold data quality checks failed.")
else:
    # Fails the Databricks notebook task if Silver fails
    raise Exception("Pipeline halted: Silver data quality checks failed. Gold layer was skipped.")